# Diplomado en Ciencia  de Datos
## Curso Series de Tiempo
### 01. Código Clase 1 Series de Tiempo

En este código se encuentran las ilustraciones y los ejemplos de la primera clase del curso de series de tiempo del diplomado en ciencia de datos

**Cristian Vásquez**


## Ilustraciones de Series de Tiempo

In [ ]:
# ----------------------------------------------------------------------------
# Instalación librerías
# ----------------------------------------------------------------------------

# install.packages("tseries")
# install.packages("forecast")
# install.packages("ggfortify")
# install.packages("fUnitRoots")


In [ ]:
# ----------------------------------------------------------------------------
# Cargar librerías
# ----------------------------------------------------------------------------

library(dplyr)
library(ggplot2)
library(scales)
library(tseries)
library(forecast)
library(ggfortify)
library(urca)
library(fUnitRoots)
library(readxl)
library(readr)

In [ ]:
# ----------------------------------------------------------------------------
# Ejemplos series de tiempos
# ----------------------------------------------------------------------------

### VAR IPC

datos = read.table("ipc_var.txt",header=T)

nc = ncol(datos)
nr = nrow(datos)

largo=nc*nr

serie = as.vector(matrix(0,largo,1))

t=1
for (i in 1:nr) {
    for (j in 1:nc) {
        serie[t]= datos[i,j]
        t       = t+1
    }
}


serie_graf = data.frame(date = seq(as.Date("1988-01-01"),as.Date("2009-12-01"), by="month"), IPC = serie)

ggplot(serie_graf, aes(x = date, y = IPC)) +
    geom_line(color = "#0072B2",size=1.1)	 +
    xlab("Fecha")                          +
    ylab("Variación del IPC")              +
    theme_minimal()


In [ ]:
### IPSA

ipsa<-read.table("ipsa.txt",header=T)

nc = ncol(ipsa)
nr = nrow(ipsa)

largo = nc*nr
serie_ipsa = as.vector(matrix(0,largo,1))
t=1
for (i in 1:nr) {
    for (j in 1:nc) {
        serie_ipsa[t] = ipsa[i,j]
        t = t+1
    }
}

datos_ipsa = data.frame(date = seq(as.Date("1990-01-01"),as.Date("2000-12-01"), by="month"), IPSA = serie_ipsa[121:252])

ggplot(datos_ipsa, aes(x = date, y = IPSA))     +
        geom_line(color = "#0072B2",size=1.1)   +
        xlab("Fecha")                           +
        ylab("IPSA")                            +
        theme_minimal()


In [ ]:
### PM2.5

pm25 <-read_excel("pm25.xlsx")

ggplot(pm25, aes(x = Fecha, y = Valor))   +
        geom_line(color = "#0072B2")      +
        xlab("Fecha")                     +
        ylab("PM 2.5")                    +
        theme_minimal()


In [ ]:
### Desocupación

desocupacion <-read_excel("desocupacion.xlsx")


ggplot(desocupacion, aes(x = Periodo, y = Tasa))  +
        geom_line(color = "#0072B2",,size=1.1)    +
        xlab("Fecha")                             +
        ylab("Tasa de Desocupación")              +
        theme_minimal()


## Evaluación Gráfica de Ruido Blanco

In [ ]:
### Ejemplo No Ruido Blanco

set.seed(123456)

rm(serie)
serie = c()

for(i in 1:144){
	serie[i] = rnorm(1,2+4*i/24,2)
}

date = seq(as.Date("2008-01-01"),as.Date("2019-12-01"), by="month")

ejemploNRB = data.frame(date = date, serie =serie, tend = 2+4*1:144/24,liminf =  (-2+4*1:144/24), limsup =  (6+4*1:144/24))


ggplot(ejemploNRB , aes(x = date, y = serie)) 					+
    geom_line(color = "#0072B2",size=1.1)    					+
    xlab("Fecha")                            					+
    ylab("Serie")  		               					+
    geom_line(aes(x = date, y =tend), color = "red", size = 1.1)			+
    geom_line(aes(x = date, y =liminf), linetype = "dashed", color = "#E69F00")	+
    geom_line(aes(x = date, y =limsup), linetype = "dashed", color = "#E69F00")	+
    ylim(0,30)								       	+
    theme_minimal()

In [ ]:
### Ejemplo No Ruido Blanco

set.seed(123456)

rm(serie)
serie = c()

for(i in 1:144){
	serie[i] = rnorm(1,0,0.5 + 5*i/144)
}

date = seq(as.Date("2008-01-01"),as.Date("2019-12-01"), by="month")

ejemploNRB = data.frame(date = date, serie =serie, liminf =  (-0.5 - 5*1:144/144)*2, limsup =  (0.5 + 5*1:144/144)*2)


ggplot(ejemploNRB , aes(x = date, y = serie)) 					                        +
    geom_line(color = "#0072B2",size=1.1)    					                          +
    xlab("Fecha")                            					                          +
    ylab("Serie")  		               					                                  +
    geom_hline(yintercept = 0, color = "red", size = 1.1)			                  +
    geom_line(aes(x = date, y =liminf), linetype = "dashed", color = "#E69F00")	+
    geom_line(aes(x = date, y =limsup), linetype = "dashed", color = "#E69F00")	+
    ylim(-11,11)								       	                                        +
    theme_minimal()


In [ ]:
### Ejemplo Ruido Blanco

serie= rnorm(12*12,0,2)
date = seq(as.Date("2008-01-01"),as.Date("2019-12-01"), by="month")

ejemploRB = data.frame(date = date, serie =serie)

ggplot(ejemploRB , aes(x = date, y = serie)) 					                  +
    geom_line(color = "#0072B2",size=1.1)    					                  +
    xlab("Fecha")                            					                  +
    ylab("Serie")  		               					                          +
    geom_hline(yintercept = 0, color = "red", size = 1.1)			          +
    geom_hline(yintercept = -4, linetype = "dashed", color = "#E69F00")	+
    geom_hline(yintercept =  4, linetype = "dashed", color = "#E69F00")	+
    ylim(-6,6)								       	                                  +
    theme_minimal()

## Función de Autocorrelación

In [ ]:
# ----------------------------------------------------------------------------
# Funcion de autocorrelación
# ----------------------------------------------------------------------------

### versión librería base stat

acf(ejemploRB$serie, main= "Ejemplo C")

In [ ]:
### versión librería forecast con ggplot

ggAcf(ejemploRB$serie,col="#0072B2",type="correlation") 	+
	ylim(-0.2,1)							+
	ggtitle("Ejemplo C")						+
	theme_minimal()

## Ejemplo: Passengers

El siguiente set de datos corresponde al número de pasajeros mensuales desde el año 2010 hasta el año 2019 de todos los aeropuertos de USA, ahí se puede diferencia por vuelos nacionales e internacionales, los datos los puede consultar en el siguiente enlace:

[https://www.transtats.bts.gov/Data_Elements.aspx?Data=1](https://)

In [ ]:
PassengersData = read_xlsx("Passengers.xlsx")

Passengers = PassengersData 									                                          %>%
              	dplyr::filter(Month != "TOTAL", Year >=2010)					                  %>%
              	dplyr::mutate(Fecha         = as.Date(paste0(Year,"-",Month,"-","01")),
              			  Domestic      = as.numeric(gsub(",","",DOMESTIC)),
              			  International = as.numeric(gsub(",","",INTERNATIONAL)),
              			  Total         = as.numeric(gsub(",","",TOTAL)))		                %>%
              	dplyr::select(Year, Month, Fecha, Domestic, International, Total)


ggplot(Passengers, aes(x = Fecha, y = International)) +
    geom_line(color = "#0072B2",size=1.1)	 		        +
    xlab("Fecha")                          		        +
    ylab("Número de Pasajeros")          			        +
    ggtitle("Pasajeros Vuelos Internacionales")		    +
    scale_y_continuous(labels = number)			          +
    theme_minimal()

In [ ]:
SerieInter = ts(Passengers$International, frequency = 12, start = c(2010,1))

Desco = decompose(SerieInter, type = "additive")

plot(decompose(SerieInter, type = "additive"))


In [ ]:
autoplot(decompose(SerieInter, type = "additive"))	+
	scale_y_continuous(labels = number)					      +
	theme_minimal()

## Análisis de los Residuos

In [ ]:
Ruido = Desco$random[!is.na(Desco$random)]

ggAcf(Ruido,col="#0072B2",type="correlation") 	+
	ylim(-0.2,1)					                        +
	ggtitle("ACF del Ruido")			                +
	theme_minimal()


## Ajuste Utilizando Regresión Lineal

In [ ]:
# ----------------------------------------------------------------------------
# Ajuste manual modelo lineal
# ----------------------------------------------------------------------------


Passengers$Tiempo = 1:117
Passengers$Month  = as.factor(Passengers$Month)
Passengers$Month  = relevel(Passengers$Month, ref = "1")

head(model.matrix(International ~ Tiempo + Month, data = Passengers),10)

modelo = lm(International ~ Tiempo + Month, data = Passengers)
summary(modelo)

Passengers$Prediccion = predict(modelo)

colors <- c("International" = "blue", "Prediccion" = "red")

ggplot(Passengers, aes(x = Fecha))                                        +
    geom_line(aes(y = International,color = "International"),size=1.1)	 	+
    geom_line(aes(y = Prediccion, color = "Prediccion"),size=1.1)	 		    +
    xlab("Fecha")                          						                    +
    ylab("Número de Pasajeros")          							                    +
    ggtitle("Pasajeros Vuelos Internacionales")						                +
    scale_y_continuous(labels = number)							                      +
    scale_color_manual(values = colors)							                      +
    labs(color = "Legend") 									                              +
    theme_minimal()


In [ ]:
# ----------------------------------------------------------------------------
# Residuos
# ----------------------------------------------------------------------------

require(gridExtra)

Passengers$Residuos = resid(modelo)

plot1 <- ggplot(Passengers, aes(Residuos))    +
          geom_histogram(alpha = 0.5,
                        fill  = "blue",
                        col   = "red",
                        bins  = 20)	          +
          theme_minimal()

plot2 <- ggplot(Passengers, aes(x = Fecha, y = Residuos)) 		+
          geom_line(color = "#0072B2",size=1.1)    			      +
          xlab("Fecha")                            			      +
          ylab("Serie")  		               			              +
          geom_hline(yintercept = 0, color = "red", size = 1.1)	+
          theme_minimal()

plot3 <- ggAcf(Passengers$Residuos,col="#0072B2",type="correlation") 	+
          ylim(-0.2,1)					                                      +
          ggtitle("ACF del Ruido")			                              +
          theme_minimal()


grid.arrange(plot1, plot2, plot3, ncol=3)

In [ ]:
ggtsdiag(Arima(Passengers$Residuos,order = c(0,0,0)), gof.lag = 20,ad.colour="blue")	+
  theme_minimal()